In [115]:
import numpy as np
import pandas as pd
import os
import sys
sys.path.append("..")
sys.path.append("../..")

# Gene data process

In [116]:
genedata = pd.read_excel('../data-Gene/all_gene_TPM.xlsx')
n_list = pd.read_excel("../data-Gene/N_P_KO_list.xlsx",sheet_name=0).values.T.tolist()[0]
p_list = pd.read_excel("../data-Gene/N_P_KO_list.xlsx",sheet_name=1).values.T.tolist()[0]

In [118]:
# 针对不同水池选出对应gene丰度
# 各个水池与n/p有关的gene丰度
n_genedata = genedata[genedata['KEGG'].isin(n_list)].iloc[:,:]
p_genedata = genedata[genedata['KEGG'].isin(p_list)].iloc[:,:]
print(n_genedata.shape,p_genedata.shape)
n_genedata.to_csv('../data-Gene/n_genedata.csv',index=False)
p_genedata.to_csv('../data-Gene/p_genedata.csv',index=False)
genedata.to_csv('../data-Gene/c_genedata.csv',index=False)

(22, 7) (96, 7)


# Wq data process

In [48]:
wqdata = pd.read_excel('../data-WQ/wq.xlsx')

In [98]:
# 去除缺失和nan
# ===================== 1. 核心插值函数（支持单列/多列、时序索引） =====================
def linear_interpolate_nan(df, cols=None, time_index=False):
    """
    两点线性插值补充NaN/缺失值
    :param df: 原始DataFrame（支持普通索引/时序索引）
    :param cols: 需要插值的列名列表（None则处理所有数值列）
    :param time_index: 是否为时序索引（True则按时间间隔插值，False按位置插值）
    :return: 插值后的DataFrame
    """
    # 复制数据避免修改原数据
    df_interp = df.copy()
    
    # 确定需要插值的列（默认所有数值列）
    if cols is None:
        cols = df_interp.select_dtypes(include=[np.number]).columns.tolist()
    
    for col in cols:
        # 跳过无NaN的列
        if not df_interp[col].isnull().any():
            continue
        
        # 时序索引：按时间间隔插值（如1小时/1天）；普通索引：按位置插值
        if time_index:
            # 确保索引是datetime类型
            df_interp.index = pd.to_datetime(df_interp.index)
            # 线性插值（method='time'适配时序）
            df_interp[col] = df_interp[col].interpolate(method='time', limit_direction='both')
        else:
            # 普通索引：两点线性插值（limit_direction='both'处理首尾NaN）
            df_interp[col] = df_interp[col].interpolate(method='linear', limit_direction='both')
    
    return df_interp

# ===================== 1. 批量一阶移动平均平滑（所有数值列） =====================
def batch_moving_average_smooth(df, window_size=int(10*60/15), time_index=True):
    """
    对所有数值列执行一阶移动平均平滑
    :param df: 原始DataFrame（含时序索引/时间列）
    :param window_size: 滑动窗口大小（默认3个时间单位）
    :param time_index: 是否为时序索引（True则按时间窗口，False按位置）
    :return: 新增所有列平滑结果的DataFrame
    """
    df_smooth = df.copy()
    # 筛选所有数值列
    num_cols = df_smooth.select_dtypes(include=[np.number]).columns.tolist()
    if not num_cols:
        raise ValueError("数据中无数值列，无法进行平滑处理")
    
    # 确保时间索引格式正确
    if time_index:
        df_smooth.index = pd.to_datetime(df_smooth.index)
    
    # 遍历所有数值列执行平滑
    smooth_col_names = []
    for col in num_cols:
        if time_index:
            # 按时间滑动窗口
            smooth_col = df_smooth[col].rolling(window=window_size, min_periods=1).mean()
        else:
            # 按位置滑动窗口
            smooth_col = df_smooth[col].rolling(window=window_size, min_periods=1).mean()
        df_smooth[f"{col}_MA_smooth"] = smooth_col
        smooth_col_names.append(f"{col}_MA_smooth")
    
    # 筛选最终列：平滑列 + 可选保留非数值列
    final_cols = smooth_col_names
    df_smooth = df_smooth[final_cols]
    
    return df_smooth

In [99]:
temwq = linear_interpolate_nan(wqdata)
wqdata1 = batch_moving_average_smooth(temwq)

In [ ]:
# 归一化
# ===================== 1. 基础实现（单列/多列归一化） =====================
def min_max_scaling(df, cols=None, feature_range=(0, 1)):
    """
    最大最小归一化（支持多列、自定义缩放区间）
    :param df: 原始DataFrame
    :param cols: 需要归一化的列名列表（None则归一化所有数值列）
    :param feature_range: 缩放区间，默认(0,1)
    :return: 归一化后的DataFrame、各列的min/max（用于后续反归一化）
    """
    # 复制数据避免修改原数据
    df_scaled = df.copy()
    # 确定需要归一化的列（默认所有数值列）
    if cols is None:
        cols = df.select_dtypes(include=[np.number]).columns.tolist()
    # 存储各列的min/max（用于反归一化）
    min_max_dict = {}
    
    min_val = feature_range[0]
    max_val = feature_range[1]
    
    for col in cols:
        # 跳过空值列（避免除以0）
        if df[col].isnull().all():
            print(f"警告：列{col}全为空，跳过归一化")
            continue
        # 计算列的最大/最小值
        col_min = df[col].min()
        col_max = df[col].max()
        min_max_dict[col] = (col_min, col_max)
        
        # 处理最大值=最小值的情况（避免除以0）
        if col_max == col_min:
            df_scaled[col] = min_val
        else:
            # 核心归一化计算
            df_scaled[col] = (df[col] - col_min) / (col_max - col_min)
            # 缩放至自定义区间（默认0-1）
            df_scaled[col] = df_scaled[col] * (max_val - min_val) + min_val
    
    return df_scaled, min_max_dict

# ===================== 3. 反归一化（还原原始数据） =====================
def inverse_min_max_scaling(df_scaled, min_max_dict, cols=None, feature_range=(0, 1)):
    """
    反归一化（将归一化数据还原为原始值）
    :param df_scaled: 归一化后的DataFrame
    :param min_max_dict: 归一化时记录的min/max字典
    :param cols: 需要反归一化的列（None则处理所有在min_max_dict中的列）
    :param feature_range: 归一化时的缩放区间
    :return: 反归一化后的DataFrame
    """
    df_original = df_scaled.copy()
    if cols is None:
        cols = min_max_dict.keys()
    
    min_val = feature_range[0]
    max_val = feature_range[1]
    
    for col in cols:
        if col not in min_max_dict:
            continue
        col_min, col_max = min_max_dict[col]
        # 反归一化计算
        df_original[col] = (df_original[col] - min_val) / (max_val - min_val) * (col_max - col_min) + col_min
    return df_original


In [109]:
wqdata_scaled, min_max_dict = min_max_scaling(wqdata1)
wqdata_scaled.to_csv('../data-WQ/wq_scaled.csv', index=False, encoding="utf-8")
np.save('../data-WQ/wq_scaleddict.npy',min_max_dict,allow_pickle=True)

In [113]:
wqdata_scaled.columns

Index(['F-INF_MA_smooth', 'T-INF_MA_smooth', 'pH-INF_MA_smooth',
       'COD-INF_MA_smooth', 'TN-INF_MA_smooth', 'AN-INF_MA_smooth',
       'TP-INF_MA_smooth', 'C/N-INF_MA_smooth', 'DO-ANA_MA_smooth',
       'AN-ANA_MA_smooth', 'NN-ANA_MA_smooth', 'DO-ANO_MA_smooth',
       'AN-ANO_MA_smooth', 'NN-ANO_MA_smooth', 'DO-OX_MA_smooth',
       'AN-OX_MA_smooth', 'NN-OX_MA_smooth', 'ORP-OX_MA_smooth',
       'AFR_MA_smooth', 'COD-EFF_MA_smooth', 'TN-EFF_MA_smooth',
       'AN-EFF_MA_smooth', 'TP-EFF_MA_smooth'],
      dtype='object')